In [32]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError
import cv2
import numpy as np

In [4]:
PHOTOS_DIR= './devastator_photos' 

In [35]:
def check_photos(file_path):
    dir = Path(file_path)
    
    if not dir.is_dir():
        print(f"Dir is not existing {dir}")
        return

    destroyed_files = []
    checked_files = 0
    good_files = 0

    extension = ".jpg"

    

    for file in dir.iterdir():
        if file.is_file() and file.suffix.lower() == extension:
            checked_files += 1
            try:
                with Image.open(file) as img:
                    img.verify() 
                good_files+=1
                
            except (UnidentifiedImageError, IOError, SyntaxError) as e:
                print(f"[!] Destroyed file: {file.name} (Cause: {e})")
                destroyed_files.append(file)
            except Exception as e:
                print(f"[!] ERROR: {file.name} (ERROR: {e})")
                destroyed_files.append(file)

    print("\n" + "="*30)
    print("SUMMARY:")
    print(f"Scanned files: {checked_files}")
    print(f"Good photos: {good_files}")
    print(f"Destroyed: {len(destroyed_files)}")
    
    if destroyed_files:
        print("\nList of destoryed_ones:")
        for file in destroyed_files:
            print(f"- {file.name}")
    return destroyed_files




In [41]:
photos_to_delete = check_photos(PHOTOS_DIR)


SUMMARY:
Scanned files: 1101
Good photos: 1101
Destroyed: 0


In [38]:
def delete_destroyed_photos(files_to_delete, dry_run=True):
    if not files_to_delete:
        print("\nNo destroyed files to delete. You are good to go!")
        return

    print("\n" + "="*30)
    if dry_run:
        print(" DRY RUN MODE ACTIVE - No files will actually be deleted.")
    else:
        print(" DANGER MODE ACTIVE - Corrupted files WILL be permanently deleted.")

    deleted_count = 0
    for file in files_to_delete:
        if dry_run:
            print(f"[DRY RUN] Would delete: {file.name}")
        else:
            try:
                file.unlink() 
                print(f"[DELETED] {file.name}")
                deleted_count += 1
            except Exception as e:
                print(f"[ERROR] Could not delete {file.name} (ERROR: {e})")
                
    if not dry_run:
        print(f"\nSuccessfully deleted {deleted_count} files.")
    else:
        print("\nTo actually delete the files, change `DRY_RUN_MODE = False` at the bottom of the script.")



In [40]:
delete_destroyed_photos(photos_to_delete,dry_run=False)


 DANGER MODE ACTIVE - Corrupted files WILL be permanently deleted.
[DELETED] zdjecie_20260410_170655_540_5.jpg
[DELETED] zdjecie_20260410_170637_175_5.jpg
[DELETED] zdjecie_20260410_170653_655_5.jpg
[DELETED] zdjecie_20260410_170633_519_5.jpg
[DELETED] zdjecie_20260410_170651_541_5.jpg
[DELETED] zdjecie_20260410_170649_088_5.jpg
[DELETED] zdjecie_20260410_170631_080_5.jpg
[DELETED] zdjecie_20260410_170628_779_5.jpg

Successfully deleted 8 files.


In [42]:
def find_object_v2(image_path, counter):
    
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Cannot load image.")
        return

    original = img.copy()
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # ── OPTIMIZED PARAMETERS ─────────────────────────────────
    # HSV ranges tuned for the yellow Devastator robot

    # Yellow HSV range (wider range for better detection)
    mask_yellow = cv2.inRange(hsv,
        np.array([12, 35, 60]),     # lower threshold (H, S, V)
        np.array([45, 255, 255]))   # upper threshold

    # Green HSV range (dark part - extended range)
    mask_green = cv2.inRange(hsv,
        np.array([30, 20, 40]),
        np.array([95, 255, 255]))

    # Dark/black part of the object (extended)
    mask_dark = cv2.inRange(hsv,
        np.array([0, 0, 0]),
        np.array([180, 100, 80]))

    # Combine all masks
    mask = cv2.bitwise_or(mask_yellow, mask_green)
    mask = cv2.bitwise_or(mask, mask_dark)

    # Morphology — remove noise, connect object fragments
    # Larger morphological kernel for better fragment merging
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (8, 8)))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        print(f"[{counter}] No objects found")
    else:
        # Increased minimum area threshold to eliminate noise
        contours = [c for c in contours if cv2.contourArea(c) > 500]
        if contours:
            # Sort by size and take the largest
            contours.sort(key=cv2.contourArea, reverse=True)
            largest = contours[0]
            x, y, w, h = cv2.boundingRect(largest)

            # Clamp bounding box to image boundaries
            x = max(0, x)
            y = max(0, y)
            w = min(w, img.shape[1] - x)
            h = min(h, img.shape[0] - y)

            cv2.rectangle(original, (x, y), (x+w, y+h), (0, 0, 255), 3)
            cropped = img[y:y+h, x:x+w]
            area = cv2.contourArea(largest)
            print(f"[{counter}] Found: bbox=({x},{y}), size={w}x{h}px, area={area:.0f}")
            cv2.imshow("Cropped object", cropped)
            cv2.imwrite(f"preprocessed_photos/object_{counter}.jpg", cropped)
        else:
            print(f"[{counter}] No objects of sufficient size found")

    def scale(im, width=700):
        r = width / im.shape[1]
        return cv2.resize(im, (width, int(im.shape[0] * r)))

    #cv2.imshow("Yellow mask",   scale(mask_yellow))
    #cv2.imshow("Green mask",    scale(mask_green))
    #cv2.imshow("Combined mask", scale(mask))
    #cv2.imshow("RESULT",        scale(original))
    #cv2.waitKey(0)
    #cv2.destroyAllWindows()


In [44]:
dir = Path(PHOTOS_DIR)
counter = 0
for file in dir.iterdir():
    counter += 1
    if file.is_file():
        try:
            with Image.open(file) as img:
                img.verify()
                # Pass file path as string instead of Path object
                find_object_v2(str(file), counter)
        except (UnidentifiedImageError, IOError, SyntaxError) as e:
            print(f"[!] Destroyed file: {file.name} (Cause: {e})")
        except Exception as e:
            print(f"[!] ERROR: {file.name} (ERROR: {e})")


Nasycenie — min: 0, max: 255, średnia: 26.3
Pikseli z S>40:  301146
Pikseli z S>20:  1863439
Znaleziono: bbox=(0,0), rozmiar=2270x150px
Nasycenie — min: 0, max: 231, średnia: 32.7
Pikseli z S>40:  587673
Pikseli z S>20:  1899967
Znaleziono: bbox=(197,322), rozmiar=1195x558px
Nasycenie — min: 0, max: 255, średnia: 34.9
Pikseli z S>40:  662416
Pikseli z S>20:  1977611
Znaleziono: bbox=(0,0), rozmiar=2304x240px
Nasycenie — min: 0, max: 255, średnia: 42.5
Pikseli z S>40:  922798
Pikseli z S>20:  2117382
Znaleziono: bbox=(359,231), rozmiar=889x281px
Nasycenie — min: 0, max: 255, średnia: 46.5
Pikseli z S>40:  935695
Pikseli z S>20:  2212253
Znaleziono: bbox=(1231,208), rozmiar=1073x576px
Nasycenie — min: 0, max: 255, średnia: 45.9
Pikseli z S>40:  1060000
Pikseli z S>20:  1911816
Znaleziono: bbox=(0,0), rozmiar=2004x1296px
Nasycenie — min: 0, max: 255, średnia: 29.1
Pikseli z S>40:  304268
Pikseli z S>20:  1975464
Znaleziono: bbox=(1455,298), rozmiar=533x998px
Nasycenie — min: 0, max: 255, 